# ESPIDAM - Networks and Contact Patterns in Infectious Disease Models
### Vittoria Colizza, Francesco Parino
(Acknowledgments: Parts of the code and exercises were developed with the help of Ronan Corgel and Shweta Bansal)

# About this practical session
## Importation risk

If a disease is detected in county A. Which are the counties at highest risk of importing a case from county A? 
The risk of importation from a county A to a county B is can be defined as the probability of traveling from A to B, conditional on traveling.  
In other words: let us assume that an infected person is about to travel out of the country. Where will they go? Importation risk to country B is the probability that they will go to B.  

So we can turn the definition of risk into a mathematical law:   
$$C_{ab} = \frac{W_{ab}}{W_a}$$

where $W_a$ is the total number of outbound travelers from county $a$ (i.e., total passengers to every destination $b$ except the origin county). The quantity $C_{ab}$ represents the probability that a traveler departing from county a arrives in county b. Notice that this measure depends only on mobility patterns and does not require epidemiological information. It therefore describes the relative distribution of importation risk across destinations rather than the absolute number of imported cases.

# Import libraries

In [2]:
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates

import os
import geopandas as gpd
import statsmodels.api as sm

In [3]:
# If you encounter an ImportError try installing packages using the following command:
# !pip install geopandas

In [4]:
# A function for formatting dates in plots
def dateFormat(ax):
    locator = mdates.AutoDateLocator(minticks=5, maxticks=10)
    formatter = mdates.ConciseDateFormatter(locator, show_offset=False)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)

# Read data

In [5]:
# Load the GeoJSON file with county coordinates
geoData = gpd.read_file('https://raw.githubusercontent.com/holtzy/The-Python-Graph-Gallery/master/static/data/US-counties.geojson')
geoData = geoData.to_crs("ESRI:102003")
geoData = geoData.set_index('id')

hideStates = ['02', '69','44' ,'66' ,'15' ,'60' ,'78' ,'72']
geoData = geoData.query("STATE not in @hideStates")

In [6]:
# Read county to county csv file
# If executing on Google Colab:
c2c = pd.read_csv('https://github.com/EPIcx-lab/ESPIDAM-2026---Networks-and-Contact-Patterns-in-Infectious-Disease-Models/raw/main/mobilityflows/mobilityFlowsCounty.csv.xz', dtype={"county_o": str, "county_d": str})
# If executing on Jupyter Lab:
c2c = pd.read_csv('./mobilityflows/mobilityFlowsCounty.csv.xz', dtype={"county_o": str, "county_d": str}) 


c2c['date'] = pd.to_datetime(c2c['date']) # transform column in datetime
c2c['county_o'] = c2c['county_o'].astype(str).apply(lambda a: a.zfill(5)) #Transform to strings and containing 5 characters
c2c['county_d'] = c2c['county_d'].astype(str).apply(lambda a: a.zfill(5)) #Transform to strings and containing 5 characters

In [7]:
# Display 10 random rows from the dataset
c2c.sample(2)

,county_o,county_d,pop_flows,date
20957909,20041,48179,24.0,2020-03-13
5561147,31185,20181,10.0,2020-04-02


# Create $C_{ab}$

In [ ]:
# Remove the travel inside the same county
c2c = c2c.query('...')

In [ ]:
# Filter mobility for a certain date
df = ... 

# Compute the denominator W_a
Wa = ...

# Add the total outgoing flow (W_a) as a new column in the dataframe
# Hint: inside the map() function, you can use a Series or DataFrame where the index holds the keys and the values are what you want to assign
...

# Now compute C_ab as a new column in the dataframe
...

# Select a source county and plot on a map the risk 

- **New York City, New York**: 36061
- **Los Angeles, California**: 06037
- **Chicago, Illinois**: 17031
- **Miami, Florida**: 12086
- **Las Vegas, Nevada**: 32003
- **Washington, D.C.**: 11001
- **Napa Valley, California**: 06055
- **Yellowstone National Park, Wyoming/Montana/Idaho**: 56029 / 30067 / 16019
- **Jackson Hole, Wyoming**: 56039
- **Lake Placid, New York**: 36031
- **Lancaster Country, Pennsylvania**: 42071
- **Hinsdale County, Colorado**: 08053
- **Great Smoky Mountains National Park, Tennessee/North Carolina**: 47009 / 37087
- **Bar Harbor, Maine**: 23009

In [ ]:
# Select the C_ab values for a specfic location and add the values to geoData 
source = '36061'  # (New York City, New York)
dfSource = ...

# Merge geoData with dfSource
geoDataC = geoData.merge(...)
geoDataC.head(2)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4), ncols=1, layout='constrained')
colorNorm = mpl.colors.LogNorm(vmin=geoDataC['C'].min(), vmax=geoDataC['C'].max())
ax.axis('off')

# Plot the whole map only borders
geoData.plot(facecolor='None', ax=ax, linewidth=0.1)

# Plot the destination color based on risk C.
geoDataC.plot(column='C', cmap='inferno_r', ax=ax, linewidth=0, vmax=0.008, legend=True, norm=colorNorm)

# Plot the source in green
geoData.loc[[source]].plot(facecolor='green', ax=ax, linewidth=0)

In [ ]:
# 2) Test other sources. Big cities vs more pheripheral counties
...

In [ ]:
# 3) Test different dates, before after lockdown
...

1. What are the consequences for incorrectly attributing an index case to an incorrect \(but nearby\) county?
2. What if an insetad of a single index case we have initial cases in multiple counties? How could you update the risk evaluation?
3. Is it easier to contain a novel infection that is imported from abroad into an urban or rural county?

# EXTRA: Compare two dates

First, let's create a function that speeds up the calculation of importation risk \(notice how this is similar to the previous section\):

In [ ]:
def addColumnC(df):
    # Compute the denominator W_a
    Wa = df.groupby('county_o')['pop_flows'].sum()
    Wa.name = 'Wa'
    
    # Add the denominator W_a to the df dataframe
    df = df.merge(Wa, left_on='county_o', right_index=True)

    # Now compute C_ab
    df['C'] = df['pop_flows']/df['Wa']
    return df

In [ ]:
# Select a certain date from the c2c DataFrame
source = ...

dfDate1 = ...
dfDate2 = ...

# Add risk column using the function addColumnC
dfDate1 = ...
dfDate2 = ...

In [ ]:
# Merge geoData with dfSource
geoDataC1 = ...
geoDataC2 = ...

In [ ]:
fig, axs = plt.subplots(figsize=(12, 4), ncols=2, layout='constrained')
colorNorm = mpl.colors.LogNorm(vmin=min(geoDataC1['C'].min(),geoDataC1['C'].min()), vmax=max(geoDataC1['C'].max(),geoDataC1['C'].max()))

# Plot the destination color based on risk C.
geoDataC1.plot(column='C', cmap='inferno_r', ax=axs[0], linewidth=0, legend=True, norm=colorNorm)
geoDataC2.plot(column='C', cmap='inferno_r', ax=axs[1], linewidth=0, legend=True, norm=colorNorm)

for ax in axs: 
    ax.axis('off')
    geoData.plot(facecolor='None', ax=ax, linewidth=0.1)
    geoData.loc[[source]].plot(facecolor='green', ax=ax, linewidth=0)

In [ ]:
# why do they look similar?

# Calculate the effective distance to predict disease arrival order better than geographic distance

Effective distance i calculated as $-log(P_{ij})$ where $P_{ij}$ is the mobility probability. This is essentially the same as $-log(C_{ab})$ that we have already calculated.



In [ ]:
### Calculate effective distance and add travel distance.

In [49]:
def addColumnC(df):
    # Compute the denominator W_a
    Wa = df.groupby('county_o')['pop_flows'].sum()
    Wa.name = 'Wa'

    # Add the denominator W_a to the df dataframe
    df = df.merge(Wa, left_on='county_o', right_index=True)

    # Now compute C_ab
    df['C'] = df['pop_flows']/df['Wa']
    return df

In [50]:
# Select a date and a source
source = '36061'
df = c2c.query('date == "2020-03-02" and county_o == @source').copy()
df = addColumnC(df)

In [51]:
df['effective_distance'] = ...

In [ ]:
# Compute the controid for each county
geoData["centroid"] = geoData.centroid

# Add centroid of the origin and destination (HINT: Use 'map()' )
df['centroid_o'] = ...
df['centroid_d'] = ...

# Drop NA rows
df = df.dropna()

# Display the data
df.head(2)

In [ ]:
# Calculate the travel distance and convert distance from meters to kilometers
centroid_o = gpd.GeoSeries(df['centroid_o'].values, crs="ESRI:102003")
centroid_d = gpd.GeoSeries(df['centroid_d'].values, crs="ESRI:102003")
df['distance'] = centroid_o.distance(centroid_d).values / 1000

# Examine 5 rows of the data
df.head(2)

### Compare effective vs. geographic distance

Examine the effective distance distribution and a scatter plot of effective vs. geographic distance.



In [ ]:
# Compare effective vs geographic distance
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Effective distance distribution
axes[0].hist(df['effective_distance'], bins=30, alpha=0.7)
axes[0].set_xlabel('Effective Distance')
axes[0].set_ylabel('Number of Counties')
axes[0].set_title('Distribution of Effective Distances from NYC')

# Plot 2: Geographic vs Effective Distance
# (Add geographic distance calculation here)
axes[1].scatter(df['distance'], df['effective_distance'], alpha=0.6)
axes[1].set_xlabel('Geographic Distance (km)')
axes[1].set_ylabel('Effective Distance')
axes[1].set_title('Geographic vs. Effective Distance')

1. What counties have the smallest geographic distance? What about effective distance?
2. How would you describe counties that have large differences between these two values?